# E-Commerce Order Analytics — Data Pipeline

This notebook handles data generation and cleaning for our e-commerce analytics project.

We'll do:
 Generate 4 realistic CSV datasets with intentional quality issues
 Clean and validate the data using Pandas
 Save cleaned files for SQL analysis in the next notebook

## Setup

In [1]:
import csv
import random
import os
import re
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker

fake = Faker()
Faker.seed(42)
random.seed(42)

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/cleaned", exist_ok=True)

print("ready to go")

ready to go


---
## Part 1: Data Generation

- ~5% orders with missing customer_id
- ~3% order items with negative quantity (returns)
- ~5% orders with wrong date format (DD-MM-YYYY instead of YYYY-MM-DD)
- ~15% product names with extra spaces or weird casing
- ~2% invalid customer emails

### Configuration

In [2]:
NUM_CUSTOMERS = 200
NUM_PRODUCTS = 80
NUM_ORDERS = 600
NUM_ORDER_ITEMS = 1500

statuses = ["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]
regions = ["US-EAST", "US-WEST", "US-CENT", "EU-WEST", "EU-EAST", "APAC"]
customer_types = ["REGULAR", "PREMIUM", "VIP"]

categories = {
    "Electronics": ["Smartphones", "Laptops", "Headphones", "Tablets", "Cameras"],
    "Clothing": ["T-Shirts", "Jeans", "Jackets", "Dresses", "Shoes"],
    "Home": ["Furniture", "Kitchen", "Lighting", "Decor", "Bedding"],
    "Books": ["Fiction", "Non-Fiction", "Academic", "Comics", "Self-Help"],
}

### Generate Customers

In [3]:
customers = []
for i in range(1, NUM_CUSTOMERS + 1):
    name = fake.name()
    email = fake.email()

    if random.random() < 0.02:
        trick = random.choice(["no_at", "no_domain", "double_at"])
        if trick == "no_at":
            email = email.replace("@", "")
        elif trick == "no_domain":
            email = email.split("@")[0]
        else:
            email = email.replace("@", "@@")

    reg_date = fake.date_between(start_date="-3y", end_date="-30d")

    customers.append({
        "customer_id": f"CUST-{i:04d}",
        "customer_name": name,
        "email": email,
        "registration_date": reg_date.strftime("%Y-%m-%d"),
        "customer_type": random.choice(customer_types),
    })

print(f"generated {len(customers)} customers")

generated 200 customers


### Generate Products

In [4]:
products = []
used_names = set()

for i in range(1, NUM_PRODUCTS + 1):
    cat = random.choice(list(categories.keys()))
    subcat = random.choice(categories[cat])

    while True:
        adj = random.choice(["Premium", "Basic", "Ultra", "Pro", "Lite",
                             "Classic", "Modern", "Elite", "Compact", "Deluxe"])
        pname = f"{adj} {subcat} {fake.word().title()}"
        if pname not in used_names:
            used_names.add(pname)
            break

    if random.random() < 0.15:
        mess = random.choice(["spaces", "case", "both"])
        if mess in ("spaces", "both"):
            pos = random.randint(0, len(pname) - 1)
            pname = pname[:pos] + "  " + pname[pos:]
        if mess in ("case", "both"):
            pname = random.choice([pname.upper(), pname.lower(), pname.swapcase()])

    products.append({
        "product_id": f"PROD-{i:04d}",
        "product_name": pname,
        "category": cat,
        "subcategory": subcat,
        "cost_price": round(random.uniform(5.0, 500.0), 2),
    })

print(f"generated {len(products)} products")

generated 80 products


### Generate Orders

In [5]:
customer_ids = [c["customer_id"] for c in customers]
orders = []
base_date = datetime(2025, 7, 1)

for i in range(1, NUM_ORDERS + 1):
    odate = base_date - timedelta(
        days=random.randint(0, 730),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59),
    )

    if random.random() < 0.05:
        date_str = odate.strftime("%d-%m-%Y %H:%M:%S")
    else:
        date_str = odate.strftime("%Y-%m-%d %H:%M:%S")

    if random.random() < 0.05:
        cid = random.choice(["NULL", ""])
    else:
        cid = random.choice(customer_ids)

    orders.append({
        "order_id": f"ORD-{i:05d}",
        "customer_id": cid,
        "order_date": date_str,
        "status": random.choice(statuses),
        "region_code": random.choice(regions),
    })

print(f"generated {len(orders)} orders")

generated 600 orders


### Generate Order Items

In [6]:
order_ids = [o["order_id"] for o in orders]
product_ids = [p["product_id"] for p in products]
items = []

for i in range(1, NUM_ORDER_ITEMS + 1):
    qty = random.randint(1, 5)
    if random.random() < 0.03:
        qty = -abs(qty)

    discount = round(random.uniform(0, 40), 2)
    if random.random() < 0.01:
        discount = round(random.uniform(100, 150), 2)

    items.append({
        "item_id": f"ITEM-{i:06d}",
        "order_id": random.choice(order_ids),
        "product_id": random.choice(product_ids),
        "quantity": qty,
        "unit_price": round(random.uniform(9.99, 999.99), 2),
        "discount_percent": discount,
    })

print(f"generated {len(items)} order items")

generated 1500 order items


### Save Raw CSVs

In [7]:
def save_csv(rows, filename):
    path = f"data/raw/{filename}"
    keys = list(rows[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    print(f"  saved {filename} ({len(rows)} rows)")

save_csv(customers, "customers.csv")
save_csv(products, "products.csv")
save_csv(orders, "orders.csv")
save_csv(items, "order_items.csv")

  saved customers.csv (200 rows)
  saved products.csv (80 rows)
  saved orders.csv (600 rows)
  saved order_items.csv (1500 rows)


### Quick look at the planted issues

In [8]:
null_custs = sum(1 for o in orders if o["customer_id"] in ("NULL", ""))
neg_qtys = sum(1 for it in items if it["quantity"] < 0)
bad_dates = sum(1 for o in orders if not o["order_date"].startswith("20"))
bad_discounts = sum(1 for it in items if it["discount_percent"] > 100)

print(f"NULL customer_ids  : {null_custs}")
print(f"Negative quantities: {neg_qtys}")
print(f"Wrong date formats : {bad_dates}")
print(f"Discount > 100%    : {bad_discounts}")

NULL customer_ids  : 20
Negative quantities: 57
Wrong date formats : 27
Discount > 100%    : 12


---
## Part 2: Data Cleaning & Validation

Now we load the raw CSVs and clean them using Pandas.

### Load raw data

In [9]:
orders_df = pd.read_csv("data/raw/orders.csv")
items_df = pd.read_csv("data/raw/order_items.csv")
products_df = pd.read_csv("data/raw/products.csv")
customers_df = pd.read_csv("data/raw/customers.csv")

print("orders:", orders_df.shape)
print("items:", items_df.shape)
print("products:", products_df.shape)
print("customers:", customers_df.shape)

orders: (600, 5)
items: (1500, 6)
products: (80, 5)
customers: (200, 5)


### clean_orders() — fix dates and handle NULL customer_ids

In [10]:
def clean_orders(df):
    df = df.copy()

    null_mask = df["customer_id"].isin(["NULL", "", "null"]) | df["customer_id"].isna()
    null_count = null_mask.sum()
    df.loc[null_mask, "customer_id"] = "UNKNOWN"

    bad_date_mask = ~df["order_date"].str.match(r"^\d{4}-\d{2}-\d{2}")
    bad_count = bad_date_mask.sum()

    correct = pd.to_datetime(df["order_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    wrong = pd.to_datetime(df["order_date"], format="%d-%m-%Y %H:%M:%S", errors="coerce")
    df["order_date"] = correct.fillna(wrong).dt.strftime("%Y-%m-%d %H:%M:%S")

    return df, {"null_customer_ids": int(null_count), "bad_date_formats": int(bad_count)}

orders_df, order_issues = clean_orders(orders_df)
print("Order issues:", order_issues)
orders_df.head()

Order issues: {'null_customer_ids': 20, 'bad_date_formats': 27}


,order_id,customer_id,order_date,status,region_code
0,ORD-00001,CUST-0112,2024-01-16 03:31:02,CANCELLED,US-EAST
1,ORD-00002,CUST-0042,2024-03-06 12:33:39,DELIVERED,EU-WEST
2,ORD-00003,CUST-0141,2023-07-21 08:41:18,PLACED,EU-WEST
3,ORD-00004,CUST-0132,2025-04-01 13:43:40,PLACED,APAC
4,ORD-00005,CUST-0194,2023-12-23 09:33:57,CANCELLED,APAC


### clean_products() — normalize names using Pandas string methods

In [11]:
def clean_products(df):
    df = df.copy()
    before = df["product_name"].copy()
    df["product_name"] = df["product_name"].str.replace(r"\s+", " ", regex=True).str.strip().str.title()
    cleaned = (before != df["product_name"]).sum()
    return df, {"names_cleaned": int(cleaned)}

products_df, product_issues = clean_products(products_df)
print("Product issues:", product_issues)
products_df.head()

Product issues: {'names_cleaned': 12}


,product_id,product_name,category,subcategory,cost_price
0,PROD-0001,Classic Dresses Response,Clothing,Dresses,39.36
1,PROD-0002,Compact Lighting Nation,Home,Lighting,490.67
2,PROD-0003,Basic Furniture Guy,Home,Furniture,134.30
3,PROD-0004,Basic Furniture Career,Home,Furniture,176.12
4,PROD-0005,Deluxe Decor Unit,Home,Decor,62.24


### validate_emails() — find invalid emails using Pandas str.match

In [12]:
def validate_emails(df):
    valid = df["email"].str.match(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
    invalid_df = df[~valid][["customer_id", "email"]]
    return invalid_df

bad_emails = validate_emails(customers_df)
print(f"Found {len(bad_emails)} customers with invalid emails:")
bad_emails

Found 4 customers with invalid emails:


,customer_id,email
67,CUST-0068,stricklandfrank@@example.com
150,CUST-0151,atkinsbarbara@@example.net
172,CUST-0173,mitchellkathrynexample.com
176,CUST-0177,marydunnexample.net


### check_referential_integrity() — find orphan order items using merge

In [13]:
def check_referential_integrity(orders_df, items_df):
    merged = items_df.merge(orders_df[["order_id"]], on="order_id", how="left", indicator=True)
    orphans = merged[merged["_merge"] == "left_only"].drop(columns="_merge")
    return orphans

orphan_items = check_referential_integrity(orders_df, items_df)
print(f"Orphan order items (referencing non-existent orders): {len(orphan_items)}")

Orphan order items (referencing non-existent orders): 0


### Clean order items and customers

In [14]:
items_df = items_df.copy()

over_100 = (items_df["discount_percent"] > 100).sum()
print(f"Discounts > 100%: {over_100} (capping to 100)")
items_df["discount_percent"] = items_df["discount_percent"].clip(upper=100)

neg_qty = (items_df["quantity"] < 0).sum()
print(f"Negative quantities (returns): {neg_qty} (flagging)")
items_df["is_return"] = items_df["quantity"] < 0

customers_df = customers_df.copy()
customers_df["customer_name"] = customers_df["customer_name"].str.strip().str.title()
customers_df["customer_type"] = customers_df["customer_type"].str.upper()
print("Customer names and types normalized")

Discounts > 100%: 12 (capping to 100)
Negative quantities (returns): 57 (flagging)
Customer names and types normalized


### Save cleaned data

In [15]:
for name, df in [("orders", orders_df), ("order_items", items_df),
                  ("products", products_df), ("customers", customers_df)]:
    df.to_csv(f"data/cleaned/{name}.csv", index=False)
    print(f"  saved cleaned {name}.csv ({len(df)} rows)")

  saved cleaned orders.csv (600 rows)
  saved cleaned order_items.csv (1500 rows)
  saved cleaned products.csv (80 rows)
  saved cleaned customers.csv (200 rows)


### Cleaning Summary

In [ ]:
print("==============================================" )
print("  DATA CLEANING SUMMARY")
print("==============================================" )
print(f"  NULL customer_ids fixed : {order_issues['null_customer_ids']}")
print(f"  Bad date formats fixed  : {order_issues['bad_date_formats']}")
print(f"  Product names cleaned   : {product_issues['names_cleaned']}")
print(f"  Invalid emails found    : {len(bad_emails)}")
print(f"  Orphan order items      : {len(orphan_items)}")
print(f"  Discounts capped (>100%): {over_100}")
print(f"  Returns flagged         : {neg_qty}")
print("==============================================")
print("\nCleaned data saved to data/cleaned/")

  DATA CLEANING SUMMARY
  NULL customer_ids fixed : 20
  Bad date formats fixed  : 27
  Product names cleaned   : 12
  Invalid emails found    : 4
  Orphan order items      : 0
  Discounts capped (>100%): 12
  Returns flagged         : 57

Cleaned data saved to data/cleaned/
